In [1]:
import pandas as pd
import numpy as np
import os

folder_path = r"D:\Swapnil\Work\Projects\FIFA\Updated Base Files"

sales = pd.read_csv(os.path.join(folder_path, "Step4_Global_Expanded.csv"))
matches = pd.read_excel(os.path.join(folder_path, "Fifa_Matches_Final.xlsx"))

print("Sales:", sales.shape)
print("Matches:", matches.shape)
print(matches.head())

Sales: (1389312, 26)
Matches: (208, 7)
  match_date match_number                                            country  \
0 2026-06-11      Match 1                                             Mexico   
1 2026-06-11      Match 1                                       South Africa   
2 2026-06-11      Match 2  Czechia/Denmark/North Macedonia/Republic of Ir...   
3 2026-06-11      Match 2                                     Korea Republic   
4 2026-06-12      Match 3  Bosnia and Herzegovina/Italy/Northern Ireland/...   

                                            opponent    group  \
0                                       South Africa  Group A   
1                                             Mexico  Group A   
2                                     Korea Republic  Group A   
3  Czechia/Denmark/North Macedonia/Republic of Ir...  Group A   
4                                             Canada  Group B   

               stadium        stage  
0  Mexico City Stadium  Group Stage  
1  Mexico Cit

In [2]:
sales["date"] = pd.to_datetime(sales["date"], errors="coerce").dt.date
matches["match_date"] = pd.to_datetime(matches["match_date"], errors="coerce").dt.date

sales["country"] = sales["country"].astype(str).str.strip()
matches["country"] = matches["country"].astype(str).str.strip()

In [3]:
match_keys = (
    matches[["match_date", "country"]]
    .dropna()
    .drop_duplicates()
    .copy()
)

match_keys["match_day_flag"] = 1

print(match_keys.head())
print(match_keys.shape)

   match_date                                            country  \
0  2026-06-11                                             Mexico   
1  2026-06-11                                       South Africa   
2  2026-06-11  Czechia/Denmark/North Macedonia/Republic of Ir...   
3  2026-06-11                                     Korea Republic   
4  2026-06-12  Bosnia and Herzegovina/Italy/Northern Ireland/...   

   match_day_flag  
0               1  
1               1  
2               1  
3               1  
4               1  
(208, 3)


In [4]:
sales = sales.merge(
    match_keys,
    left_on=["date", "country"],
    right_on=["match_date", "country"],
    how="left"
)

sales["match_day_flag"] = sales["match_day_flag"].fillna(0).astype(int)

sales = sales.drop(columns=["match_date"], errors="ignore")

print(sales["match_day_flag"].value_counts())

match_day_flag
0    1389312
Name: count, dtype: int64


In [5]:
match_count = (
    matches.groupby("country")
    .size()
    .reset_index(name="match_count")
)

sales = sales.merge(match_count, on="country", how="left")
sales["match_count"] = sales["match_count"].fillna(0).astype(int)

In [6]:
sales["fifa_total_sales"] = sales["sim_total_sales"]
sales["fifa_units_sold"] = sales["sim_units_sold"]
sales["fifa_operating_profit"] = sales["sim_operating_profit"]

In [7]:
match_mask = sales["match_day_flag"] == 1

sales.loc[match_mask, "fifa_total_sales"] *= 1.35
sales.loc[match_mask, "fifa_units_sold"] *= 1.25
sales.loc[match_mask, "fifa_operating_profit"] *= 1.30

In [8]:
print(matches["stage"].value_counts())

stage
Group Stage       144
Knockout/Other     64
Name: count, dtype: int64


In [9]:
knockout_keys = (
    matches[matches["stage"].astype(str).str.contains("Knockout", case=False, na=False)]
    [["match_date", "country"]]
    .drop_duplicates()
)

knockout_keys["knockout_stage_flag"] = 1

sales = sales.merge(
    knockout_keys,
    left_on=["date", "country"],
    right_on=["match_date", "country"],
    how="left"
)

sales["knockout_stage_flag"] = sales["knockout_stage_flag"].fillna(0).astype(int)
sales = sales.drop(columns=["match_date"], errors="ignore")

knockout_mask = sales["knockout_stage_flag"] == 1

sales.loc[knockout_mask, "fifa_total_sales"] *= 1.50
sales.loc[knockout_mask, "fifa_operating_profit"] *= 1.40

In [10]:
sales["knockout_stage_flag"] = 0

In [11]:
sales["fifa_sales_uplift_pct"] = (
    (sales["fifa_total_sales"] - sales["sim_total_sales"])
    / sales["sim_total_sales"]
) * 100

sales["fifa_sales_uplift_pct"] = sales["fifa_sales_uplift_pct"].replace(
    [np.inf, -np.inf], 0
).fillna(0)

In [12]:
print("Final shape:", sales.shape)

print(sales[[
    "date",
    "country",
    "company_name",
    "sim_total_sales",
    "match_day_flag",
    "knockout_stage_flag",
    "fifa_total_sales",
    "fifa_sales_uplift_pct"
]].head(20))

print("Match day rows:")
print(sales["match_day_flag"].value_counts())

print("Average uplift:")
print(sales["fifa_sales_uplift_pct"].mean())

Final shape: (1389312, 33)
          date                 country company_name  sim_total_sales  \
0   2022-01-01                  Canada    Coca-Cola           9000.0   
1   2022-01-01                  Mexico    Coca-Cola           9375.0   
2   2022-01-01           United States    Coca-Cola           9750.0   
3   2022-01-01               Argentina    Coca-Cola           7500.0   
4   2022-01-01                  Brazil    Coca-Cola           7800.0   
5   2022-01-01                Colombia    Coca-Cola           6300.0   
6   2022-01-01                 Ecuador    Coca-Cola           6000.0   
7   2022-01-01                Paraguay    Coca-Cola           5100.0   
8   2022-01-01                 Uruguay    Coca-Cola           6000.0   
9   2022-01-01                 Austria    Coca-Cola           6000.0   
10  2022-01-01                 Belgium    Coca-Cola           6600.0   
11  2022-01-01  Bosnia and Herzegovina    Coca-Cola           5100.0   
12  2022-01-01                 Croati

In [13]:
output_path = os.path.join(folder_path, "Step5_Global_With_FIFA.csv")

sales.to_csv(output_path, index=False)

print("Saved:", output_path)

Saved: D:\Swapnil\Work\Projects\FIFA\Updated Base Files\Step5_Global_With_FIFA.csv
